# Deep Learning with PyTorch · Sequences: RNN, LSTM and attention

Macademia · For adults

> **Running in Colab:** Open this file at [colab.research.google.com](https://colab.research.google.com) with **File → Upload notebook**. For a GPU choose **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom (Shift + Enter). PyTorch is preinstalled in Colab; to run on your own computer use `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # reproducibility
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. Task: remember the sign at the start of the sequence
The sign of the first step decides the label; the rest is noise. On long sequences an RNN struggles.

In [ ]:
from torch import nn
def veri(n, T):
    x = torch.randn(n, T, 1) * 0.5
    y = (x[:, 0, 0] > 0).long()
    x[:, 0, 0] = torch.where(y == 1, 1.0, -1.0)
    return x, y
class Model(nn.Module):
    def __init__(self, kind='lstm', h=32):
        super().__init__()
        self.rnn = (nn.LSTM if kind == 'lstm' else nn.RNN)(1, h, batch_first=True); self.fc = nn.Linear(h, 2)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1])

## 2. RNN vs LSTM

In [ ]:
def egit(kind, T, steps=300):
    torch.manual_seed(0); m = Model(kind).to(device); opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    for _ in range(steps):
        x, y = veri(64, T); x, y = x.to(device), y.to(device)
        opt.zero_grad(); l = nn.functional.cross_entropy(m(x), y); l.backward()
        nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
    x, y = veri(500, T)
    with torch.no_grad():
        return (m(x.to(device)).argmax(1).cpu() == y).float().mean().item()
for T in [10, 50, 100]:
    print(f'T={T:3d}  RNN {egit("rnn", T):.2f}  LSTM {egit("lstm", T):.2f}')

## 3. Scaled dot-product attention

In [ ]:
import torch.nn.functional as F
Q = K_ = V = torch.randn(1, 5, 16)
w = torch.softmax(Q @ K_.transpose(-2, -1) / 16 ** 0.5, dim=-1)
print(w.sum(-1))
print(torch.allclose(w @ V, F.scaled_dot_product_attention(Q, K_, V), atol=1e-6))
mha = nn.MultiheadAttention(16, num_heads=4, batch_first=True)
out, att = mha(Q, K_, V)
print(out.shape, att.shape)

## Exercises
1. Run the same comparison with `nn.GRU`.
2. Remove gradient clipping; what happened to the RNN?
3. Plot the attention weights as a heatmap.